# News Topic Classifier Using BERT

Fine-tune `bert-base-uncased` on the **AG News** dataset to classify headlines into 4 categories:
- World
- Sports
- Business
- Science/Technology

> **Runtime:** Go to `Runtime -> Change runtime type -> T4 GPU`

# Step 1: Fix torchvision + datasets compatibility, then install all deps
import subprocess, sys

# 1. Uninstall conflicting packages
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision', 'datasets'], capture_output=True)

# 2. Reinstall torchvision WITHOUT video extras (removes VideoReader dependency)
!pip install -q torchvision --no-deps

# 3. Install pinned datasets version that does not probe torchvision.io.VideoReader
!pip install -q 'datasets==2.19.2'

# 4. Install remaining dependencies
!pip install -q transformers scikit-learn gradio accelerate

print('All packages installed.')
print('>>> IMPORTANT: Click Runtime -> Restart session, then run from Step 2 <<<')

In [1]:
# Pin datasets to avoid the torchvision VideoReader import bug
!pip install -q 'numpy<2.0'
!pip install -q 'datasets==2.19.2' transformers scikit-learn gradio accelerate
# Restart runtime after this cell if running for the first time
import importlib, datasets
print('datasets version:', datasets.__version__)

datasets version: 2.19.2


## Step 2: Import Libraries

In [2]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    pipeline,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Using device: cuda
GPU: Tesla T4


## Step 3: Load and Explore the AG News Dataset

In [3]:
# Using fully-qualified path for reliability across datasets versions
dataset = load_dataset('fancyzhx/ag_news')
print(dataset)

LABEL_NAMES = ['World', 'Sports', 'Business', 'Science/Technology']
LABEL_MAP   = {i: name for i, name in enumerate(LABEL_NAMES)}

print('\nLabel mapping:')
for idx, name in LABEL_MAP.items():
    print(f'  {idx} -> {name}')

print('\nSample training examples:')
for i in range(3):
    ex = dataset['train'][i]
    print(f"  [{LABEL_MAP[ex['label']]}] {ex['text'][:120]}...")

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

Label mapping:
  0 -> World
  1 -> Sports
  2 -> Business
  3 -> Science/Technology

Sample training examples:
  [Business] Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics,...
  [Business] Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\which has a reputat...
  [Business] Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\about the economy and the ou...


## Step 4: Tokenize the Dataset

In [4]:
MODEL_NAME = 'bert-base-uncased'
MAX_LEN    = 128

tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
    )

# Set USE_SUBSET=False to train on full dataset (~30 min on T4)
USE_SUBSET = True

if USE_SUBSET:
    train_raw = dataset['train'].shuffle(seed=42).select(range(12000))
    test_raw  = dataset['test'].shuffle(seed=42).select(range(2000))
else:
    train_raw = dataset['train']
    test_raw  = dataset['test']

train_tok = train_raw.map(tokenize, batched=True, remove_columns=['text'])
test_tok  = test_raw.map(tokenize,  batched=True, remove_columns=['text'])

train_tok = train_tok.rename_column('label', 'labels')
test_tok  = test_tok.rename_column('label',  'labels')

train_tok.set_format('torch')
test_tok.set_format('torch')

print(f'Train size : {len(train_tok):,}')
print(f'Test  size : {len(test_tok):,}')
print(f'Columns    : {train_tok.column_names}')

Train size : 12,000
Test  size : 2,000
Columns    : ['labels', 'input_ids', 'token_type_ids', 'attention_mask']


## Step 5: Load BERT and Set Up Metrics

In [5]:
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label=LABEL_MAP,
    label2id={v: k for k, v in LABEL_MAP.items()},
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average='weighted')
    return {'accuracy': acc, 'f1': f1}

print('Model loaded:', model.config.architectures[0])
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: BertForMaskedLM
Parameters: 109,485,316


## Step 6: Fine-Tune with Hugging Face Trainer

In [6]:
# Calculate warmup_steps manually (replaces deprecated warmup_ratio)
EPOCHS     = 3
BATCH_SIZE = 32
num_training_steps  = (len(train_tok) // BATCH_SIZE) * EPOCHS
num_warmup_steps    = int(0.1 * num_training_steps)  # 10% warmup
print(f'Total training steps : {num_training_steps}')
print(f'Warmup steps         : {num_warmup_steps}')

training_args = TrainingArguments(
    output_dir                  = './bert-ag-news',
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = 64,
    warmup_steps                = num_warmup_steps,  # fixed: no longer uses warmup_ratio
    weight_decay                = 0.01,
    learning_rate               = 2e-5,
    eval_strategy         = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    logging_steps               = 50,
    fp16                        = torch.cuda.is_available(),
    report_to                   = 'none',
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_tok,
    eval_dataset    = test_tok,
    processing_class       = tokenizer,
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
)

print('Starting fine-tuning...')
trainer.train()

Total training steps : 1125
Warmup steps         : 112
Starting fine-tuning...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.283115,0.270341,0.905500,0.905568
2,0.171316,0.253437,0.919500,0.919535


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.283115,0.270341,0.905500,0.905568
2,0.171316,0.253437,0.919500,0.919535
3,0.118523,0.267841,0.921500,0.921466


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=1125, training_loss=0.27736337916056314, metrics={'train_runtime': 356.2691, 'train_samples_per_second': 101.047, 'train_steps_per_second': 3.158, 'total_flos': 1951463962027008.0, 'train_loss': 0.27736337916056314, 'epoch': 3.0})

In [7]:
import numpy as np
print('NumPy version:', np.__version__)

NumPy version: 1.26.4


## Step 7: Evaluate — Accuracy, F1, and Full Report

In [8]:
print('Evaluating on test set...')
results = trainer.evaluate()

print(f"\n{'='*40}")
print(f"  Accuracy : {results['eval_accuracy']:.4f}")
print(f"  F1 Score : {results['eval_f1']:.4f}")
print(f"{'='*40}\n")

preds_output = trainer.predict(test_tok)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids

print('Per-class Classification Report:')
print(classification_report(y_true, y_pred, target_names=LABEL_NAMES))

Evaluating on test set...


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.118523,0.267841,3,0.921500,0.921466



  Accuracy : 0.9215
  F1 Score : 0.9215



Per-class Classification Report:
                    precision    recall  f1-score   support

             World       0.93      0.93      0.93       497
            Sports       0.98      0.98      0.98       483
          Business       0.90      0.87      0.89       522
Science/Technology       0.88      0.90      0.89       498

          accuracy                           0.92      2000
         macro avg       0.92      0.92      0.92      2000
      weighted avg       0.92      0.92      0.92      2000



## Step 8: Save the Model

In [9]:
SAVE_DIR = './bert-ag-news-final'
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'Model saved to {SAVE_DIR}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./bert-ag-news-final


## Step 9: Quick Inference Test

In [ ]:
classifier = pipeline(
    'text-classification',
    model=SAVE_DIR,
    tokenizer=SAVE_DIR,
    device=0 if torch.cuda.is_available() else -1,
)

test_headlines = [
    'Scientists discover new exoplanet with potential for life',
    'Stock markets rally after Federal Reserve signals rate pause',
    'Manchester City wins Premier League title in dramatic finale',
    'UN Security Council meets to address escalating border tensions',
    'Apple unveils next-generation chip with record AI performance',
]

print('Inference Results')
print('-' * 55)
for headline in test_headlines:
    result = classifier(headline)[0]
    print(f'Headline : {headline}')
    print(f'Predicted: {result["label"]}  (confidence: {result["score"]:.2%})')
    print()

## Step 10: Gradio Interactive Demo

In [11]:
import gradio as gr

def classify_news(headline):
    if not headline.strip():
        return {label: 0.0 for label in LABEL_NAMES}
    outputs = classifier(
        headline,
        top_k=None,
        truncation=True,
        max_length=MAX_LEN,
    )
    return {o['label']: float(o['score']) for o in outputs}

examples = [
    ['Scientists discover water on Mars in groundbreaking study'],
    ['Tesla reports record quarterly earnings beating analyst expectations'],
    ['LeBron James scores 40 points to lead Lakers to playoff victory'],
    ['G20 leaders meet to discuss global climate change commitments'],
    ['New AI model passes bar exam with 90 percent accuracy'],
]

demo = gr.Interface(
    fn          = classify_news,
    inputs      = gr.Textbox(
                    label='News Headline',
                    placeholder='Paste or type a news headline here...',
                    lines=2,
                  ),
    outputs     = gr.Label(num_top_classes=4, label='Category Probabilities'),
    title       = 'News Topic Classifier (BERT)',
    description = 'Fine-tuned bert-base-uncased on AG News. Categories: World, Sports, Business, Science/Technology',
    examples    = examples,
    theme       = gr.themes.Soft(),
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://921912fbbbb985cb41.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
